# PUMA ARC Solver + LLM Integration
## Full Pipeline with LLM Meta-Reasoning Layer

In [ ]:
import os
import sys
from pathlib import Path
import json
import gc
from IPython.display import display, HTML, clear_output
import time
import numpy as np

print("🚀 PUMA ARC Solver + LLM Meta-Reasoning")
print("="*60)

# Paths
KAGGLE_INPUT_DIR = Path("/kaggle/input/arc-prize-2025")
KAGGLE_DATASET_DIR = Path("/kaggle/input/puma11/public2")
ARC_SOLVER_PATH = KAGGLE_DATASET_DIR / "arc_solver"

# Fix paths and imports
if str(KAGGLE_DATASET_DIR) not in sys.path:
    sys.path.insert(0, str(KAGGLE_DATASET_DIR))
if str(ARC_SOLVER_PATH) not in sys.path:
    sys.path.insert(0, str(ARC_SOLVER_PATH))

import arc_solver.rft_engine as puma_module
sys.modules['puma'] = puma_module
print("✓ Import paths configured")

TEST_CHALLENGES_PATH = KAGGLE_INPUT_DIR / "arc-agi_test_challenges.json"
FAST_MEMORY_PATH = KAGGLE_DATASET_DIR / "fast_comprehensive_memory.json"
MODEL_PATH = KAGGLE_DATASET_DIR / "models" / "phi-3-mini-4k-instruct"

os.environ["ARC_ENABLE_LOGGING"] = "false"
print(f"✓ Model: {MODEL_PATH.name}")
print("="*60)

## Progress GUI

In [ ]:
class ProgressPanel:
    def __init__(self, total_tasks):
        self.total_tasks = total_tasks
        self.completed = 0
        self.successful = 0
        self.failed = 0
        self.current_task = ""
        self.start_time = time.time()
        self.last_process = "Initializing..."
        self.llm_calls = 0
        
    def update(self, task_id="", process="", success=None, llm_used=False):
        if task_id:
            self.current_task = task_id
        if process:
            self.last_process = process
        if llm_used:
            self.llm_calls += 1
        
        if success is not None:
            self.completed += 1
            if success:
                self.successful += 1
            else:
                self.failed += 1
        
        elapsed = time.time() - self.start_time
        avg_time = elapsed / self.completed if self.completed > 0 else 0
        remaining = (self.total_tasks - self.completed) * avg_time
        progress_pct = (self.completed / self.total_tasks * 100) if self.total_tasks > 0 else 0
        
        html = f"""
        <div style="border: 2px solid #4CAF50; border-radius: 10px; padding: 20px; 
                    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    color: white; font-family: 'Courier New', monospace;">
            <h2 style="text-align: center; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">🐆 PUMA + 🧠 LLM</h2>
            <div style="background: rgba(0,0,0,0.3); padding: 15px; border-radius: 5px; margin: 10px 0;">
                <div style="font-size: 18px; margin-bottom: 10px;">Progress: {self.completed}/{self.total_tasks}</div>
                <div style="background: rgba(255,255,255,0.2); height: 30px; border-radius: 15px; overflow: hidden;">
                    <div style="background: linear-gradient(90deg, #4CAF50, #8BC34A); height: 100%; width: {progress_pct}%; transition: width 0.3s;"></div>
                </div>
                <div style="margin-top: 5px;">{progress_pct:.1f}%</div>
            </div>
            <div style="display: grid; grid-template-columns: 1fr 1fr 1fr 1fr; gap: 10px; margin: 10px 0;">
                <div style="background: rgba(76,175,80,0.3); padding: 10px; border-radius: 5px; text-align: center;">
                    <div style="font-size: 24px;">✓ {self.successful}</div>
                    <div style="font-size: 11px;">Success</div>
                </div>
                <div style="background: rgba(244,67,54,0.3); padding: 10px; border-radius: 5px; text-align: center;">
                    <div style="font-size: 24px;">✗ {self.failed}</div>
                    <div style="font-size: 11px;">Failed</div>
                </div>
                <div style="background: rgba(255,152,0,0.3); padding: 10px; border-radius: 5px; text-align: center;">
                    <div style="font-size: 24px;">🧠 {self.llm_calls}</div>
                    <div style="font-size: 11px;">LLM</div>
                </div>
                <div style="background: rgba(33,150,243,0.3); padding: 10px; border-radius: 5px; text-align: center;">
                    <div style="font-size: 24px;">{remaining/60:.0f}m</div>
                    <div style="font-size: 11px;">ETA</div>
                </div>
            </div>
            <div style="background: rgba(0,0,0,0.3); padding: 12px; border-radius: 5px;">
                <div style="font-size: 11px; opacity: 0.7;">TASK</div>
                <div style="font-size: 14px; font-weight: bold; color: #FFD700;">{self.current_task or 'N/A'}</div>
                <div style="font-size: 11px; opacity: 0.7; margin-top: 8px;">PROCESS</div>
                <div style="font-size: 13px; color: #4CAF50;">{self.last_process}</div>
            </div>
        </div>
        """
        clear_output(wait=True)
        display(HTML(html))

print("✓ Progress Panel Ready")

## LLM-Enhanced Solver Wrapper

In [ ]:
from arc_solver import ARCSolver
from arc_solver.llm_interface import LLMInterface, LLMConfig
from arc_solver.llm_meta_reasoner import LLMMetaReasoner
from arc_solver.features import extract_task_features
from arc_solver.grid import to_array

class LLMEnhancedSolver:
    """Wrapper that adds LLM meta-reasoning to ARCSolver."""
    
    def __init__(self, model_path, memory_path, enable_llm=True):
        print("⚙️  Initializing LLM-Enhanced Solver...\n")
        
        # Initialize base solver
        print("🐆 Loading PUMA ARCSolver...")
        self.solver = ARCSolver(
            use_enhancements=True,
            episode_db_path=str(memory_path),
            enable_logging=False
        )
        print("✓ ARCSolver ready")
        
        # Initialize LLM meta-reasoner
        self.enable_llm = enable_llm
        self.llm_calls = 0
        
        if enable_llm:
            print(f"\n🧠 Loading Phi-3 model from {model_path.name}...")
            llm_config = LLMConfig(
                model_name=str(model_path),
                use_4bit=True,
                temperature=0.7,
                max_tokens=512
            )
            self.llm_interface = LLMInterface(config=llm_config)
            self.meta_reasoner = LLMMetaReasoner(
                llm_config=llm_config,
                enabled=True
            )
            print("✓ LLM Meta-Reasoner ready")
        else:
            self.llm_interface = None
            self.meta_reasoner = None
        
        print("\n" + "="*60)
        print("✅ SYSTEM READY")
        print("="*60)
        print("Active Features:")
        if enable_llm:
            print("  🧠 LLM Meta-Reasoning (Phi-3)")
        print("  📚 RFT Engine")
        print("  🔍 Enhanced Search")
        print("  💭 Hypothesis Engine")
        print("  📖 Episodic Memory")
        print("="*60)
    
    def solve_task(self, task_data):
        """Solve task with optional LLM guidance."""
        
        # Extract training pairs
        train_pairs = []
        for pair in task_data.get('train', []):
            try:
                inp = to_array(pair['input'])
                out = to_array(pair['output'])
                train_pairs.append((inp, out))
            except:
                continue
        
        # LLM Meta-Reasoning (optional)
        if self.enable_llm and self.meta_reasoner and train_pairs:
            try:
                # Extract features for LLM
                task_features = extract_task_features(train_pairs)
                
                # Get episodic memories
                similar_episodes = self.solver.episodic_retrieval.retrieve(
                    task_features, k=5
                )
                
                # Get LLM guidance
                meta_result = self.meta_reasoner.reason_about_task(
                    train_pairs=train_pairs,
                    task_features=task_features,
                    similar_episodes=similar_episodes,
                    rft_facts=[],  # Could populate from RFT engine
                    predicted_ops=[]  # Could get from neural guidance
                )
                
                self.llm_calls += 1
                # LLM gave us strategy guidance, now run solver
                
            except Exception as e:
                # LLM failed, just run solver normally
                pass
        
        # Run the actual solver (with all enhancements)
        result = self.solver.solve_task(task_data)
        return result

# Initialize
llm_solver = LLMEnhancedSolver(
    model_path=MODEL_PATH,
    memory_path=FAST_MEMORY_PATH,
    enable_llm=True
)

## Solve All Tasks

In [ ]:
print("\n📂 Loading test challenges...")
with open(TEST_CHALLENGES_PATH, 'r') as f:
    test_challenges = json.load(f)
print(f"✓ Loaded {len(test_challenges)} tasks\n")

panel = ProgressPanel(len(test_challenges))
panel.update(process="Starting solver...")

submission = {}
task_list = list(test_challenges.items())

for idx, (task_id, task_data) in enumerate(task_list):
    panel.update(task_id=task_id, process=f"Task {idx+1}/{len(task_list)}")
    
    try:
        # Solve with LLM + PUMA
        panel.update(process="Running solver...")
        result = llm_solver.solve_task(task_data)
        
        # Check if LLM was used
        llm_used = (llm_solver.llm_calls > panel.llm_calls)
        if llm_used:
            panel.update(process="✓ LLM + Solver", llm_used=True)
        
        # Extract results
        attempt1 = result.get('attempt_1', [])
        attempt2 = result.get('attempt_2', [])
        
        num_test = len(task_data.get('test', []))
        output_list = []
        
        for i in range(num_test):
            test_input = task_data['test'][i]['input']
            output_list.append({
                "attempt_1": attempt1[i] if i < len(attempt1) else test_input,
                "attempt_2": attempt2[i] if i < len(attempt2) else test_input
            })
        
        submission[task_id] = output_list
        panel.update(process="✓ Complete", success=True)
        
    except Exception as e:
        # Fallback
        output_list = [
            {"attempt_1": tc['input'], "attempt_2": tc['input']}
            for tc in task_data.get('test', [])
        ]
        submission[task_id] = output_list
        panel.update(process=f"✗ Error", success=False)
    
    if (idx + 1) % 10 == 0:
        gc.collect()

# Save
with open("submission.json", "w") as f:
    json.dump(submission, f)

elapsed = time.time() - panel.start_time
print("\n" + "="*60)
print("🎉 COMPLETE!")
print("="*60)
print(f"✓ Success: {panel.successful}/{panel.total_tasks} ({panel.successful/panel.total_tasks*100:.1f}%)")
print(f"🧠 LLM Calls: {panel.llm_calls}")
print(f"⏱️  Time: {elapsed/60:.1f}min")
print("="*60)